# 📥 LangChain Inputs and Outputs

## Learning Objectives
In this notebook, you will learn:
1. **Single invocation** - use `.invoke()` for one request and read the `AIMessage` it returns
2. **Batch processing** - use `.batch()` for a list of prompts and `.generate()` for a list of conversations
3. **Message types** - build a conversation from `SystemMessage`, `HumanMessage` and `AIMessage` (or role dicts)
4. **Conversation history** - append each reply so the model can see earlier turns
5. **Batched conversations** - run several conversations, each with its own system prompt, in one call
6. **Response handling** - read text and token usage from `AIMessage` and `LLMResult`

## Where this fits
Module 01 called chat models through LangChain. This notebook looks at the call itself: what you
can pass in and what comes back. `02_LLM_vs_ChatModel.ipynb` then explains why a model accepts both
a plain string and a list of messages.

## Prerequisites
- `modules/01_getting_started/notebooks/03_Commercial_LLMs_with_LangChain.ipynb` (reading)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

## 🔑 Key Concepts

| Method | Input | Output | Use Case |
|--------|-------|--------|----------|
| `.invoke()` | String or Messages | AIMessage | Single request |
| `.batch()` | List of Strings | List of AIMessages | Multiple simple requests |
| `.generate()` | List of Message Lists | LLMResult | Multiple conversations with metadata |

---
## 📦 Step 1: Environment Setup

Load the API key from the project-root `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: one chat model for every call in this notebook
# ============================================================================
from aied_helpers import get_openai_llm

# temperature=0.7 keeps the jokes below from repeating word for word
llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 💬 Part 1: Single invocation

`.invoke()` is the simplest way to call a model: one prompt in, one `AIMessage` out. The text of
the reply is on `.content`.

In [ ]:
# ============================================================================
# BASIC LLM INVOCATION: The .invoke() Method
# ============================================================================
# The .invoke() method is the simplest way to interact with an LLM
# - Input: A string prompt (or list of messages)
# - Output: An AIMessage object containing the response
# ============================================================================

result = llm.invoke("Tell me a joke")

# The response is an AIMessage object - use .content to get the text
print("Response type:", type(result))
print("Response content:", result.content)

---
## 📦 Part 2: Batch processing

When you need to process **multiple prompts**, LangChain provides two approaches:

1. **`.batch()`** - simple and clean, accepts a list of strings
2. **`.generate()`** - more control, expects a list of message lists

`.batch()` runs the calls concurrently (a thread pool), so a list of prompts finishes faster than
a loop of `.invoke()` calls. Each prompt is still its own API request.

In [ ]:
# ============================================================================
# METHOD 1: Using .batch() - The Simple Approach
# ============================================================================
# .batch() accepts a list of string prompts and returns a list of AIMessage objects
# This is the recommended approach for simple batch processing
# ============================================================================

prompts = [
    "Tell me a joke about cows",
    "Tell me a joke about parrots"
]

# Process multiple prompts in one call
batch_results = llm.batch(prompts)

# Extract and display each response
print("=" * 60)
print("BATCH PROCESSING RESULTS")
print("=" * 60)
for i, result in enumerate(batch_results):
    print(f"\n📝 Prompt {i+1}: {prompts[i]}")
    print(f"🤖 Response: {result.content}")
    print("-" * 60)

In [ ]:
# ============================================================================
# METHOD 2: Using .generate() - More Control & Metadata
# ============================================================================
# .generate() is more powerful but requires structured input:
# - Input: List of message lists (each inner list is a conversation)
# - Output: LLMResult object with generations, token usage, and metadata
# 
# Use .generate() when you need:
# - Access to token usage statistics
# - Full metadata about each generation
# - Complex multi-turn conversations in batch
# ============================================================================

from langchain_core.messages import HumanMessage

# Each inner list represents a separate conversation/request
generate_result = llm.generate([
    [HumanMessage(content="Tell me a joke about cows")],
    [HumanMessage(content="Tell me a joke about parrots")]
])

# The result contains detailed information about each generation
print("Type of result:", type(generate_result))
print("\nNumber of generations:", len(generate_result.generations))

In [ ]:
# ============================================================================
# INSPECT THE GENERATIONS: one entry per input conversation
# ============================================================================
# generations is a list of lists: outer = one per conversation,
# inner = one per completion (n=1 by default)
# ============================================================================
for i, generations in enumerate(generate_result.generations):
    first = generations[0]
    print(f"[{i}] {type(first).__name__}: {first.text[:80]}")

---
## 🗣️ Part 3: Message types

LangChain provides three core message types for building conversations:

| Message Type | Purpose | Example |
|--------------|---------|---------|
| **SystemMessage** | Sets the AI's behavior/personality | "You are a helpful restaurant assistant" |
| **HumanMessage** | User's input/questions | "What's on the menu?" |
| **AIMessage** | AI's previous responses | "We have pasta, pizza, and salads" |

**Why use message types?**
- Gives the LLM context about who said what
- Allows multi-turn conversations with history
- System messages guide the AI's behavior throughout the conversation

In [ ]:
# ============================================================================
# CREATING A MULTI-TURN CONVERSATION
# ============================================================================
# This example simulates a restaurant chatbot conversation
# We build the conversation history manually to show how the LLM maintains context
# ============================================================================

from langchain_core.messages import HumanMessage,SystemMessage,AIMessage

# Build a conversation with history
# The order matters: System → Human → AI → Human → ...
messages = [
    # 1. System message: Sets the assistant's role and personality
    SystemMessage(content="You are a helpful assistant specialized in providing information about BellaVista Italian Restaurant."),
    
    # 2. First user question
    HumanMessage(content="What's on the menu?"),
    
    # 3. AI's previous response (simulated history)
    AIMessage(content="BellaVista offers a variety of Italian dishes including pasta, pizza, and seafood."),
    
    # 4. Follow-up question from user
    HumanMessage(content="Do you have vegan options?")
]

# Display the conversation structure
print("📋 Conversation Structure:")
print("-" * 50)
for msg in messages:
    role = type(msg).__name__.replace("Message", "")
    print(f"{role:>8}: {msg.content[:60]}{'...' if len(msg.content) > 60 else ''}")

In [ ]:
# ============================================================================
# INSPECTING MESSAGE OBJECTS
# ============================================================================
# Each message is a Pydantic object with:
# - content: The actual text
# - additional_kwargs: Extra parameters (like function calls)
# - response_metadata: Metadata about the response (for AI messages)
# ============================================================================

# View the raw message objects (useful for debugging)
print("📦 Raw Message Objects:")
for i, msg in enumerate(messages):
    print(f"\n[{i}] {type(msg).__name__}:")
    print(f"    content: {msg.content[:200]}...")

In [ ]:
# ============================================================================
# INVOKING LLM WITH CONVERSATION HISTORY
# ============================================================================
# When you pass a list of messages, the LLM:
# 1. Reads the SystemMessage to understand its role
# 2. Reviews the conversation history (Human/AI exchanges)
# 3. Generates a contextually appropriate response
# ============================================================================

# The LLM will respond to the last HumanMessage with full context
llm_result = llm.invoke(input=messages)

print("🤖 AI Response to 'Do you have vegan options?':")
print("-" * 50)
print(llm_result.content)
print("-" * 50)
print(f"\n📊 Token usage: {llm_result.usage_metadata}")

### 3.1 The dict spelling

Messages can also be OpenAI-style dictionaries with `role` and `content` keys. LangChain converts
each dict to the matching message class before the call, so both spellings send the same request.

In [ ]:
# ============================================================================
# DICT SPELLING: OpenAI-style role/content dictionaries
# ============================================================================
from langchain_core.messages import convert_to_messages

prompt = "Can you explain what is Generative AI in 3 bullet points?"
sys_prompt = "Act as a helpful assistant and give meaningful examples in your responses."

dict_messages = [
    {"role": "system", "content": sys_prompt},
    {"role": "user", "content": prompt},
]

# What LangChain turns the dicts into before calling the model
for msg in convert_to_messages(dict_messages):
    print(f"{type(msg).__name__:>13}: {msg.content[:60]}")

response = llm.invoke(dict_messages)
print("\n🤖 AI Response:")
print(response.content)

---
## 🧵 Part 4: Conversation history

A chat model keeps no state between calls. To continue a conversation, append the model's reply
(an `AIMessage`) and the next user message to the list, and send the whole list again.

In [ ]:
# ============================================================================
# BUILDING CONVERSATION HISTORY: append the reply, then the follow-up
# ============================================================================
history = convert_to_messages(dict_messages)
history.append(response)  # the AIMessage the model just returned
history.append(HumanMessage(content="What did we discuss so far?"))

print("📋 Conversation history:")
for i, msg in enumerate(history):
    role = type(msg).__name__.replace("Message", "")
    preview = msg.content[:40] + "..." if len(msg.content) > 40 else msg.content
    print(f"  [{i}] {role}: {preview}")

# The model answers the follow-up from the turns in the list
response = llm.invoke(history)
print("\n🤖 AI Response (with conversation context):")
print(response.content)

The answer to "What did we discuss so far?" can only summarise Generative AI because the
earlier turns are in the list. Drop them and the model has nothing to summarise.

---
## 🌍 Part 5: Batch processing with conversations

You can also batch process **entire conversations** (not just simple prompts). This is useful for:
- **Parallel translations** - Same content, different system prompts
- **A/B testing prompts** - Compare different phrasings
- **Multi-language support** - Translate to multiple languages simultaneously

The `.generate()` method is particularly useful here as it provides detailed metadata for each generation.

In [ ]:
# ============================================================================
# EXAMPLE: Parallel Translation with Different System Prompts
# ============================================================================
# Here we translate the same phrase to multiple languages simultaneously
# Each inner list is a complete conversation with its own SystemMessage
# ============================================================================

# Create two separate conversations with different translation instructions
batch_messages = [
    # Conversation 1: English → German
    [
        SystemMessage(content="You are a helpful assistant that translates English to German. Only respond with the translation, nothing else."),
        HumanMessage(content="Do you have vegan options?")
    ],
    # Conversation 2: English → Spanish
    [
        SystemMessage(content="You are a helpful assistant that translates English to Spanish. Only respond with the translation, nothing else."),
        HumanMessage(content="Do you have vegan options?")
    ],
]

# Generate responses for both conversations in parallel
batch_result = llm.generate(batch_messages)

# Display results
print("🌍 Parallel Translation Results:")
print("=" * 50)
print(f"🇬🇧 Original: 'Do you have vegan options?'")
print(f"🇩🇪 German:   {batch_result.generations[0][0].text}")
print(f"🇪🇸 Spanish:  {batch_result.generations[1][0].text}")
print("=" * 50)

---
## 🔍 Part 6: Extracting data from LLMResult

The `.generate()` method returns an `LLMResult` object with rich metadata:
- **generations**: List of generated responses (one per input conversation)
- **llm_output**: Aggregate info like total token usage
- **model_dump()**: Convert to dictionary for easy inspection

> **Coming up:** `05_Output_Parser.ipynb` turns model text into structured Python objects.

In [ ]:
# ============================================================================
# INSPECTING LLMResult STRUCTURE
# ============================================================================
# .model_dump() converts the Pydantic object to a dictionary - useful for
# debugging, logging responses, and extracting specific metadata
# ============================================================================
result_dict = batch_result.model_dump()

print("📦 LLMResult Structure:")
print("-" * 40)
for key in result_dict:
    print(f"  • {key}")

# Per conversation: each generation's message carries its own usage_metadata
print("\n📊 Token usage per conversation:")
for i, generations in enumerate(batch_result.generations):
    usage = generations[0].message.usage_metadata or {}
    print(f"  [{i}] input={usage.get('input_tokens', 'N/A')}, "
          f"output={usage.get('output_tokens', 'N/A')}, total={usage.get('total_tokens', 'N/A')}")

# Whole call: llm_output sums the provider's token counts across conversations
token_usage = (batch_result.llm_output or {}).get("token_usage", {})
print(f"\n📊 Whole batch: total_tokens={token_usage.get('total_tokens', 'N/A')}")

In [ ]:
# ============================================================================
# EXTRACTING RESPONSES FROM LLMResult
# ============================================================================
# The generations attribute is a nested list:
# - Outer list: One entry per input conversation
# - Inner list: Multiple generations if n > 1 (default is 1)
# - Each generation has .text for the content
# ============================================================================

# Extract just the text from each generation
translations = [generation[0].text for generation in batch_result.generations]

# Display final results
print("✅ Extracted Translations:")
languages = ["German", "Spanish"]
for lang, translation in zip(languages, translations):
    print(f"  {lang}: {translation}")

---
## 📝 Summary

In this notebook, we learned:

### 1. Single invocation
- `.invoke()` sends one prompt (or message list) and returns a single `AIMessage`

### 2. Batch processing
- `.batch()` takes a list of string prompts and returns a list of `AIMessage`s; the calls run concurrently
- `.generate()` takes a list of message lists (full conversations) and returns an `LLMResult`

### 3. Message types
- `SystemMessage`, `HumanMessage` and `AIMessage` give the model roles and history; order matters
- Role/content dicts are an equivalent spelling that LangChain converts to message objects

### 4. Conversation history
- Chat models keep no state: append each reply and the next question, then send the whole list

### 5. Batched conversations
- Each inner list of `.generate()` is its own conversation, so each can carry its own system prompt

### 6. Response handling
- `AIMessage.content` holds the text; `AIMessage.usage_metadata` holds token counts
- `LLMResult.generations` is a nested list (one entry per conversation); `llm_output["token_usage"]` sums the batch

### Next Steps
- **`02_LLM_vs_ChatModel.ipynb`**: the two model interfaces behind "string or messages"